In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
!nvidia-smi

Tue Sep 22 13:34:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content
!git clone https://github.com/Syed1611/curvature-tuning-research.git
%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

!git pull

/content
fatal: destination path 'curvature-tuning-research' already exists and is not an empty directory.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 15 (delta 8), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 29.46 KiB | 685.00 KiB/s, done.
From https://github.com/Syed1611/curvature-tuning-research
   bcf1e12..8ac2649  stage-wise-ct -> origin/stage-wise-ct
Updating bcf1e12..8ac2649
Fast-forward
 ...Masters_Research_testing_on_Curve_Tunings.ipynb | 12716 +++----------------
 .../generalization_stagewise_full_ct.py            |   484 +
 src/curvature-tuning/utils/curvature_tuning.py     |   180 +
 3 files changed, 2397 insertions(+), 10983 deletions(-)
 create mode 100644 src/curvature-tuning/generalization_stagewise_full_ct.py


In [ ]:
%pip install -q -r requirements.txt

%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━

The reason behind downgrading the numpy version is because of the datasets and loguru library not being compitble with the numpy version preinstalled in the oldest environment in colab. It'll ask to restart just restart and then run the code below this.
#**`DO NOT RUN THE CODE ABOVE AFTER YOU CLICK ON RESTART`**

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


In [ ]:
from datasets import load_dataset

data_files = {
    "train": "hf://datasets/AI-Lab-Makerere/beans/data/train-00000-of-00001.parquet",
    "validation": "hf://datasets/AI-Lab-Makerere/beans/data/validation-00000-of-00001.parquet",
    "test": "hf://datasets/AI-Lab-Makerere/beans/data/test-00000-of-00001.parquet",
}

beans = load_dataset("parquet", data_files=data_files)

print(beans)
print("Train:", len(beans["train"]))
print("Validation:", len(beans["validation"]))
print("Test:", len(beans["test"]))
print("Columns:", beans["train"].column_names)
print(beans["train"].features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})
Train: 1034
Validation: 133
Test: 128
Columns: ['image_file_path', 'image', 'labels']
{'image_file_path': Value(dtype='string', id=None), 'image': Image(mode=None, decode=True, id=None), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'], id=None)}


In [ ]:
import sys

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Train samples: 1034
Validation samples: 133
Test samples: 128


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
!nvidia-smi

Tue Sep 22 13:23:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content
!git clone https://github.com/Syed1611/curvature-tuning-research.git
%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 185 (delta 48), reused 166 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 1.12 MiB | 3.22 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct


#There will be few errors while/after installing few packages, just ignore them.

In [ ]:
%pip install -q -r requirements.txt

%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━

The reason behind downgrading the numpy version is because of the datasets and loguru library not being compitble with the numpy version preinstalled in the oldest environment in colab. It'll ask to restart just restart and then run the code below this.
#**`DO NOT RUN THE CODE ABOVE AFTER YOU CLICK ON RESTART`**

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


In [ ]:
from datasets import load_dataset

data_files = {
    "train": "hf://datasets/AI-Lab-Makerere/beans/data/train-00000-of-00001.parquet",
    "validation": "hf://datasets/AI-Lab-Makerere/beans/data/validation-00000-of-00001.parquet",
    "test": "hf://datasets/AI-Lab-Makerere/beans/data/test-00000-of-00001.parquet",
}

beans = load_dataset("parquet", data_files=data_files)

print(beans)
print("Train:", len(beans["train"]))
print("Validation:", len(beans["validation"]))
print("Test:", len(beans["test"]))
print("Columns:", beans["train"].column_names)
print(beans["train"].features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})
Train: 1034
Validation: 133
Test: 128
Columns: ['image_file_path', 'image', 'labels']
{'image_file_path': Value(dtype='string', id=None), 'image': Image(mode=None, decode=True, id=None), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'], id=None)}


In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os

print(os.getcwd())
!ls

/content/curvature-tuning-research/src/curvature-tuning
/content/curvature-tuning-research/src/curvature-tuning
approximate_gelu.py
beta_acc_trend.py
beta_coeff_stats.py
compare_generalization_results_ablation.py
compare_generalization_results_gelu.py
compare_generalization_results.py
compare_generalization_results_tuned_lora.py
compare_robustness_results.py
compare_robustness_trainable_ct_results.py
demo_activation.py
demo_activation_sub.py
demo_classification_gif.py
demo_classification.py
demo_regression_gif.py
demo_regression.py
demo_toy_example_gif.py
demo_toy_example.py
generalization_ablation_ct.py
generalization_ct.py
generalization_lp_for_gelu.py
generalization_stagewise_ct.py
generalization_trainable_ct_for_gelu.py
generalization_trainable_ct.py
generalization_tuned_lora.py
README.md
requirements.txt
robustness.py
robustness_trainable_ct.py
submit_generalization_ablation_ct.py
submit_generalization_ct.py
submit_generalization_ct_vgg.py
submit_generalization_lp_for_gelu.py
subm

In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Train samples: 1034
Validation samples: 133
Test samples: 128


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


In [ ]:
!cat results/base_imagenet_to_beans_resnet18_seed42.json
!cat results/ct_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 1539,
  "accuracy": 89.0625
}{
  "num_params": 1539,
  "accuracy": 91.40625,
  "beta": 0.78,
  "coeff": 0.5,
  "best_val_acc": 97.74436090225564,
  "val_acc_list": [
    90.97744360902256,
    90.22556390977444,
    94.73684210526316,
    93.98496240601504,
    96.2406015037594,
    95.48872180451127,
    95.48872180451127,
    96.2406015037594,
    97.74436090225564,
    96.99248120300751,
    96.2406015037594,
    96.99248120300751,
    95.48872180451127,
    95.48872180451127,
    94.73684210526316,
    93.23308270676692,
    93.23308270676692,
    94.73684210526316,
    93.23308270676692,
    92.4812030075188,
    92.4812030075188,
    92.4812030075188,
    94.73684210526316,
    92.4812030075188,
    93.98496240601504,
    93.98496240601504,
    89.47368421052632,
    91.72932330827068,
    93.98496240601504,
    91.72932330827068,
    33.08270676691729
  ]
}

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8

2026-09-22 13:39:06.497 | INFO     | __main__:main:241 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42.log
2026-09-22 13:39:06.559 | INFO     | __main__:main:257 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 131MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 13:39:08.384 | INFO     | __main__:main:308 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.8

2026-09-22 13:47:10.372 | INFO     | __main__:main:241 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43.log
2026-09-22 13:47:10.432 | INFO     | __main__:main:257 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 13:47:11.807 | INFO     | __main__:main:308 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 13:47:11.808 | INFO     | __main__:main:324 - Trainable curvature parameters: 4
2026-09-22 13:47:11.808 | INFO     | __main__:

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.8

2026-09-22 13:50:38.885 | INFO     | __main__:main:241 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44.log
2026-09-22 13:50:38.990 | INFO     | __main__:main:257 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 13:50:40.257 | INFO     | __main__:main:308 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 13:50:40.258 | INFO     | __main__:main:324 - Trainable curvature parameters: 4
2026-09-22 13:50:40.258 | INFO     | __main__:

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
betas = []

for seed in seeds:
    path = (
        f"results/"
        f"stage_ct_imagenet_to_beans_resnet18_seed{seed}.json"
    )

    with open(path) as f:
        result = json.load(f)

    accs.append(result["accuracy"])
    betas.append(result["stage_betas"])

print("Stage-Wise CT")
print("Accuracies:", accs)
print(f"Mean accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Stage-Wise CT
Accuracies: [89.84375, 90.625, 90.625]
Mean accuracy: 90.36%
Std: 0.37

Stage betas:
42 [0.8327, 0.9485, 0.9086, 0.6446]
43 [0.8478, 0.9478, 0.8341, 0.6126]
44 [0.8484, 0.9421, 0.8545, 0.6361]

Mean stage betas: [0.843  0.9461 0.8657 0.6311]


#Testing out with learning rate of betas in 0.03 to test if it gives new results or similar to previous.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8 \
    --beta_lr 0.03

2026-09-22 14:03:08.238 | INFO     | __main__:main:243 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42betalr0.03.log
2026-09-22 14:03:08.297 | INFO     | __main__:main:259 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:03:09.372 | INFO     | __main__:main:310 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 14:03:09.372 | INFO     | __main__:main:326 - Trainable curvature parameters: 4
2026-09-22 14:03:09.373 | INFO     |

# Trying out beta learning rate as 0.01 to test out again because learning rate of 0.03 gave same results as the previous ones without learning rate.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-22 14:08:52.662 | INFO     | __main__:main:243 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42betalr0.01.log
2026-09-22 14:08:52.726 | INFO     | __main__:main:259 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:08:53.798 | INFO     | __main__:main:310 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 14:08:53.798 | INFO     | __main__:main:326 - Trainable curvature parameters: 4
2026-09-22 14:08:53.799 | INFO     |

#Here the accuracy has increased a bit for seed 42 so we are going to calculate it for seed 43, and 44.

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-22 14:14:44.187 | INFO     | __main__:main:243 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43betalr0.01.log
2026-09-22 14:14:44.263 | INFO     | __main__:main:259 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:14:45.364 | INFO     | __main__:main:310 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 14:14:45.365 | INFO     | __main__:main:326 - Trainable curvature parameters: 4
2026-09-22 14:14:45.365 | INFO     |

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.8 \
    --beta_lr 0.01

2026-09-22 14:18:23.438 | INFO     | __main__:main:243 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44betalr0.01.log
2026-09-22 14:18:23.534 | INFO     | __main__:main:259 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:18:24.914 | INFO     | __main__:main:310 - Initial stage betas: [0.800000011920929, 0.800000011920929, 0.800000011920929, 0.800000011920929]
2026-09-22 14:18:24.914 | INFO     | __main__:main:326 - Trainable curvature parameters: 4
2026-09-22 14:18:24.915 | INFO     |

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]
accs = []
val_accs = []
betas = []

for seed in seeds:
    path = (
        f"results/stage_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}.jsonbetalr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Test accuracies: [90.625, 89.84375, 90.625]
Mean test accuracy: 90.36%
Std: 0.37

Best validation accuracies: [96.99248120300751, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.74%

Stage betas:
42 [0.8186, 0.8834, 0.8391, 0.6617]
43 [0.8152, 0.8941, 0.8102, 0.654]
44 [0.8229, 0.8846, 0.8173, 0.6435]

Mean stage betas: [0.8189 0.8874 0.8222 0.6531]


#The stage wise results are unchanged so stopping the beta learing rate and starting with initialization of beta starting at 0.7

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-22 14:27:23.709 | INFO     | __main__:main:244 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42betalr0.01.log
2026-09-22 14:27:23.853 | INFO     | __main__:main:260 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:27:24.943 | INFO     | __main__:main:311 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 14:27:24.943 | INFO     | __main__:main:327 - Trainable curvature parameters: 4
2026-09-22 14:27:24.944 | INFO  

#We got a bit better results from the last trainings so we are going to continue with this

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-22 14:32:15.924 | INFO     | __main__:main:244 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43betalr0.01.log
2026-09-22 14:32:15.985 | INFO     | __main__:main:260 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:32:17.048 | INFO     | __main__:main:311 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 14:32:17.049 | INFO     | __main__:main:327 - Trainable curvature parameters: 4
2026-09-22 14:32:17.049 | INFO  

In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.78 \
    --beta_lr 0.01

2026-09-22 14:35:44.547 | INFO     | __main__:main:244 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44betalr0.01.log
2026-09-22 14:35:44.606 | INFO     | __main__:main:260 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 14:35:45.630 | INFO     | __main__:main:311 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 14:35:45.630 | INFO     | __main__:main:327 - Trainable curvature parameters: 4
2026-09-22 14:35:45.631 | INFO  

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
val_accs = []
betas = []

for seed in seeds:
    path = (
        f"results/stage_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}.json"
        f"initbeta0.78_"
        f"betalr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "\nMean stage betas:",
    np.round(np.mean(np.array(betas), axis=0), 4)
)

Test accuracies: [90.625, 90.625, 91.40625]
Mean test accuracy: 90.89%
Std: 0.37

Best validation accuracies: [97.74436090225564, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.99%

Stage betas:
42 [0.8154, 0.8787, 0.8259, 0.6574]
43 [0.8115, 0.8875, 0.8022, 0.6448]
44 [0.8152, 0.875, 0.8071, 0.6382]

Mean stage betas: [0.814  0.8804 0.8117 0.6468]


In [ ]:
!WANDB_MODE=disabled python generalization_trainable_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42

2026-09-22 14:43:49.348 | INFO     | __main__:main:94 - Log file: ./logs/generalization_trainable_ct_imagenet_to_beans_resnet18_seed42.log
2026-09-22 14:43:49.352 | INFO     | __main__:main:98 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-22 14:43:50.468 | INFO     | __main__:main:124 - Testing Trainable CT...
2026-09-22 14:43:50.865 | INFO     | __main__:main:129 - Number of trainable parameters: 5507
2026-09-22 14:43:50.994 | INFO     | __main__:main:131 - Mean Beta: 0.799953, Mean Coeff: 0.500000
2026-09-22 14:43:50.995 | INFO     | __main__:main:132 - Sta

In [ ]:
!cat results/train_ct_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 5507,
  "accuracy": 90.625,
  "beta": 0.7232918739318848,
  "coeff": 0.6046077013015747
}

In [ ]:
!cat results/lora_rank1_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 37462,
  "accuracy": 94.53125
}

#Transition from 4-Parameter to 8-Parameter Stage-Wise Curvature Tuning

The initial Stage-Wise Curvature Tuning experiment used four trainable curvature parameters: one shared β for each major ResNet-18 stage, while the CTU mixing coefficient \(c\) was fixed at 0.5. The learned β values showed consistent stage-specific behavior across seeds, but the method remained slightly below S-CT in mean test accuracy.

We now extend the method to 8 trainable curvature parameters. Each ResNet stage receives one trainable β and one trainable \(c\):

Stage 1: β₁, c₁

Stage 2: β₂, c₂

Stage 3: β₃, c₃

Stage 4: β₄, c₄

This maintains stage-level parameter sharing while allowing both curvature and CTU blending behavior to adapt independently across network depth. The experiment is intended to test whether the additional stage-level flexibility improves performance while retaining far fewer curvature parameters than full T-CT.

To implement this we are going to add a new class in the code which can caluclate with these 8 parameters and do a test run with all seed and check.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-22 15:14:46.985 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed42_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-22 15:14:47.047 | INFO     | __main__:main:282 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-22 15:14:48.133 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 15:14:48.134 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-22 

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-22 15:19:50.372 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed43_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-22 15:19:50.436 | INFO     | __main__:main:282 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-22 15:19:51.502 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 15:19:51.503 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-22 

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_full_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --init_beta 0.78 \
    --init_coeff 0.5 \
    --ct_lr 0.01

2026-09-22 15:23:31.983 | INFO     | __main__:main:269 - Log file: ./logs/stage_full_ct_imagenet_to_beans_resnet18_seed44_initbeta0.78_initc0.5_ctlr0.01.log
2026-09-22 15:23:32.046 | INFO     | __main__:main:282 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise Full CT ReLU counts: [3, 2, 2, 2]
2026-09-22 15:23:33.142 | INFO     | __main__:main:338 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 15:23:33.143 | INFO     | __main__:main:343 - Initial stage coeffs: [0.5, 0.5, 0.5, 0.5]
2026-09-22 

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

accs = []
val_accs = []
betas = []
coeffs = []

for seed in seeds:
    path = (
        f"results/stage_full_ct_imagenet_to_beans_"
        f"resnet18_seed{seed}_"
        f"initbeta0.78_initc0.5_ctlr0.01.json"
    )

    with open(path) as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])
    coeffs.append(r["stage_coeffs"])

print("Test accuracies:", accs)
print(f"Mean test accuracy: {np.mean(accs):.2f}%")
print(f"Std: {np.std(accs):.2f}")

print("\nBest validation accuracies:", val_accs)
print(f"Mean best validation accuracy: {np.mean(val_accs):.2f}%")

print("\nStage betas:")
for seed, b in zip(seeds, betas):
    print(seed, [round(x, 4) for x in b])

print(
    "Mean stage betas:",
    np.round(np.mean(betas, axis=0), 4)
)

print("\nStage coeffs:")
for seed, c in zip(seeds, coeffs):
    print(seed, [round(x, 4) for x in c])

print(
    "Mean stage coeffs:",
    np.round(np.mean(coeffs, axis=0), 4)
)

Test accuracies: [89.84375, 91.40625, 91.40625]
Mean test accuracy: 90.89%
Std: 0.74

Best validation accuracies: [96.99248120300751, 96.2406015037594, 96.99248120300751]
Mean best validation accuracy: 96.74%

Stage betas:
42 [0.8007, 0.8675, 0.8347, 0.6568]
43 [0.7935, 0.8819, 0.8173, 0.6379]
44 [0.8006, 0.864, 0.8179, 0.6411]
Mean stage betas: [0.7983 0.8711 0.8233 0.6453]

Stage coeffs:
42 [0.6678, 0.6609, 0.3381, 0.523]
43 [0.6965, 0.6046, 0.3573, 0.5606]
44 [0.6613, 0.675, 0.3609, 0.5011]
Mean stage coeffs: [0.6752 0.6468 0.3521 0.5282]


# Now we are trying on different Dataset: DTD

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_dtd",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/content/curvature-tuning-research/src/curvature-tuning
Downloading/extracting DTD...


100%|██████████| 625M/625M [00:33<00:00, 18.7MB/s]


Train: 1880
Validation: 1880
Test: 1880
Classes: 47


In [3]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_dtd",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train:", len(train_loader.dataset))
print("Validation:", len(val_loader.dataset))
print("Test:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Min label:", labels.min().item())
print("Max label:", labels.max().item())

Train: 1880
Validation: 1880
Test: 1880


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Min label: 1
Max label: 45


#Running the orignal S-CT on DTD dataset on seed 42

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-22 17:21:40.968 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed42.log
2026-09-22 17:21:40.971 | INFO     | __main__:main:58 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 162MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-22 17:21:41.931 | INFO     | __main__:main:85 - Testing baseline...
2026-09-22 17:21:41.945 | INFO     | __main__:main:88 - Number of trainable parameters: 24111
202

#Running the SW-CT on DTD dataset on seed 42

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-22 17:48:10.847 | INFO     | __main__:main:244 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42betalr0.01.log
2026-09-22 17:48:10.910 | INFO     | __main__:main:260 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-22 17:48:11.425 | INFO     | __main__:main:311 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-22 17:48:11.425 | INFO     | __main__:main:327 - Trainable curvature parameters: 4
2026-09-22 17:48:11.426 | INFO    